# 📓 Semana 12 · Dia 1 — Avaliação de RAG com mlflow.evaluate (LLM-as-judge)

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | GenAI Engineer Associate (Evaluation ~20%) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Métricas ≥ alvo do RAG |

---


## 📖 Teoria — As 4 métricas de ouro do RAG

| Métrica | Mede | Significado |
|---|---|---|
| **Faithfulness** | resposta segue o contexto? | 1 = sem alucinação |
| **Answer relevance** | resposta responde a pergunta? | 1 = relevante |
| **Context precision** | chunks relevantes vêm primeiro? | ordem do retrieval |
| **Context recall** | o contexto tinha tudo que precisava? | cobertura |

**LLM-as-judge**: um LLM avalia as respostas do seu RAG (usando FMA) — escalável e barato. O `mlflow.evaluate` automatiza tudo.


### 💻 Na prática — Preparando o dataset de avaliação

Crie um dataset com perguntas e respostas esperadas (golden set).


In [ ]:
# Golden set (perguntas + resposta de referência)
import pandas as pd
eval_df = pd.DataFrame({
    "question": [
        "Quais produtos são de vidro?",
        "Qual o produto mais vendido?",
        "Existe item para cozinha?"
    ],
    "answer": [
        "Não informado ainda.",
        "Preciso consultar os dados.",
        "Há itens de cozinha no catálogo."
    ]
})
print(eval_df)

In [ ]:
# Gerar respostas do RAG para o dataset
respostas = []
for q in eval_df["question"]:
    r = rag.invoke({"input": q})
    respostas.append(r["answer"])
eval_df["response"] = respostas
eval_df["retrieved_context"] = eval_df["question"].apply(
    lambda q: [d.page_content for d in rag.invoke({"input": q})["context"]])
print(eval_df[["question", "response"]])

### 💻 Na prática — Rodando a avaliação

Avalie com mlflow.evaluate usando LLM-as-judge.


In [ ]:
# Avaliação com métricas de RAG
import mlflow
with mlflow.start_run(run_name="avaliacao_rag_v1"):
    resultado = mlflow.evaluate(
        data=eval_df[["question", "response", "retrieved_context"]],
        targets=eval_df["answer"],
        model_type="databricks-agent",
        extra_metrics=[mlflow.metrics.genai.faithfulness(),
                       mlflow.metrics.genai.answer_relevance(),
                       mlflow.metrics.genai.context_precision(),
                       mlflow.metrics.genai.context_recall()])
    print("Métricas:", {k: round(v, 3) for k, v in resultado.metrics.items() if isinstance(v, float)})

In [ ]:
# Interpretar
print("""
Alvo: faithfulness >= 0.9 (pouca alucinação)
      answer relevance >= 0.8
      context recall/precision >= 0.8
Se algo caiu, revise chunking, prompt ou rerank (Semana 12.3).
""")
print("Cada métrica tem feedback do juiz na run do MLflow.")

> 🎯 **Dica de prova**: GenAI Assoc (Evaluation & Monitoring ~20%): as 4 métricas de ouro e o LLM-as-judge são perguntas garantidas. Memorize o que cada uma mede.


## 🎯 Exercícios de fixação

**1.** O que indica faithfulness baixa?

**2.** Crie mais 5 perguntas no golden set e reavalie.

**3.** Por que usar um golden set (respostas de referência)?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Faithfulness baixa

O modelo está alucinando — resposta fora do contexto. Mitigue com prompt mais restritivo e melhor retrieval.

**2.** Golden set

Quanto maior o golden set, mais confiável a métrica (20–50 perguntas é um bom começo).

**3.** Referência

Permite medir answer relevance/recall contra o esperado — sem ele, só medimos consistência.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*